# 02 — Qualidade dos Dados e Transformação para a Camada Silver

Nesta etapa, são avaliados os principais aspectos de qualidade dos dados armazenados na camada Bronze, com foco em completude, unicidade da chave país-ano e consistência dos atributos categóricos. O objetivo é identificar problemas que possam impactar as etapas posteriores de integração e análise, sem modificar os dados originais.

1. Carregamento da Bronze

In [0]:
# Importar as funções utilizadas 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Carregar as tabelas persistidas na camada Bronze
df_who = spark.table("workspace.bronze.who_life_expectancy")
df_idh = spark.table("workspace.bronze.idh")

print(f"WHO: {df_who.count()} linhas e {len(df_who.columns)} colunas")
print(f"IDH: {df_idh.count()} linhas e {len(df_idh.columns)} colunas")

WHO: 2938 linhas e 10 colunas
IDH: 5336 linhas e 5 colunas


In [0]:
# Verificar os tipos dos atributos na camada Bronze
print("SCHEMA WHO")
df_who.printSchema()

print("\nSCHEMA IDH")
df_idh.printSchema()

SCHEMA WHO
root
 |-- pais: string (nullable = true)
 |-- ano: string (nullable = true)
 |-- status: string (nullable = true)
 |-- exp_vida: string (nullable = true)
 |-- mortalidade_adulta: string (nullable = true)
 |-- mortalidade_infantil: string (nullable = true)
 |-- mortalidade_hiv: string (nullable = true)
 |-- consumo_alcool: string (nullable = true)
 |-- IMC: string (nullable = true)
 |-- pib_dolar: string (nullable = true)


SCHEMA IDH
root
 |-- paises: string (nullable = true)
 |-- ano: string (nullable = true)
 |-- anos_escolaridade_media: string (nullable = true)
 |-- indice_educacao: string (nullable = true)
 |-- IDH: string (nullable = true)



2. Diagnóstico de qualidade

In [0]:
# Valores ausentes
def resumo_ausencias(df):
    total = df.count()

    dados = []

    for coluna in df.columns:
        texto = F.trim(F.col(coluna).cast("string"))

        qtd = df.filter(
            F.col(coluna).isNull()
            | (texto == "")
            | (F.upper(texto) == "NA")
        ).count()

        dados.append(
            (coluna, qtd, round(qtd / total * 100, 2))
        )

    return spark.createDataFrame(
        dados,
        ["atributo", "ausentes", "percentual"]
    )

display(resumo_ausencias(df_who))
display(resumo_ausencias(df_idh))

atributo,ausentes,percentual
pais,0,0.0
ano,0,0.0
status,0,0.0
exp_vida,10,0.34
mortalidade_adulta,10,0.34
mortalidade_infantil,0,0.0
mortalidade_hiv,0,0.0
consumo_alcool,194,6.6
IMC,34,1.16
pib_dolar,448,15.25


atributo,ausentes,percentual
paises,0,0.0
ano,0,0.0
anos_escolaridade_media,236,4.42
indice_educacao,236,4.42
IDH,4236,79.39


In [0]:
# Duplicatas
duplicatas_who = (
    df_who.groupBy("pais", "ano")
    .count()
    .filter(F.col("count") > 1)
)

duplicatas_idh = (
    df_idh.groupBy("paises", "ano")
    .count()
    .filter(F.col("count") > 1)
)

print("Chaves duplicadas WHO:", duplicatas_who.count())
print("Chaves duplicadas IDH:", duplicatas_idh.count())

display(duplicatas_idh)

Chaves duplicadas WHO: 0
Chaves duplicadas IDH: 58


paises,ano,count
China,2018,2
China,2017,2
China,2016,2
China,2015,2
China,2014,2
China,2013,2
China,2012,2
China,2011,2
China,2010,2
China,2009,2


In [0]:
#Verificando
display(df_who.groupBy("status").count())

status,count
Developing,2426
Developed,512


3. Transformação Silver

In [0]:
# Converter valores numéricos armazenados como texto para DOUBLE
def numero(coluna):
    return F.expr(f"""try_cast(regexp_replace(trim(`{coluna}`), ',', '.') AS DOUBLE)""")

In [0]:
# Padronizando nomes, convertento os tipos e preparando a base WHO para a camada Silver
# WHO
df_who_silver = (
    df_who
    .select(
        F.trim("pais").alias("pais"),
        F.col("ano").cast("int").alias("ano"),
        F.trim("status").alias("status"),
        numero("exp_vida").alias("exp_vida"),
        numero("mortalidade_adulta").alias("mortalidade_adulta"),
        numero("mortalidade_infantil").alias("mortalidade_infantil"),
        numero("mortalidade_hiv").alias("mortalidade_hiv"),
        numero("consumo_alcool").alias("consumo_alcool"),
        numero("IMC").alias("imc"),
        numero("pib_dolar").alias("pib_dolar")
    )
    .withColumn(
        "flag_registro_invalido",
        F.when(
            (~F.col("status").isin("Developed", "Developing"))
            | (F.col("ano") < 2000)
            | (F.col("ano") > 2015)
            | (F.col("exp_vida") <= 0),
            1
        ).otherwise(0)
    )
)

In [0]:
# Padronizando nomes, convertento os tipos e preparando a base IDH para a camada Silver
# IDH
df_idh_silver = (
    df_idh
    .select(
        F.trim("paises").alias("pais"),
        F.col("ano").cast("int").alias("ano"),
        numero("anos_escolaridade_media").alias("anos_escolaridade_media"),
        numero("indice_educacao").alias("indice_educacao"),
        numero("IDH").alias("idh")
    )
)

In [0]:
# Define uma janela por país e ano para identificar duplicidades da chave
janela = Window.partitionBy("pais", "ano")

# Cria flags para duplicatas e valores do índice de educação fora do esperado
df_idh_silver = (
    df_idh_silver
    .withColumn(
        "flag_chave_duplicada",
        F.when(F.count("*").over(janela) > 1, 1).otherwise(0)
    )
    .withColumn(
        "flag_indice_educacao_invalido",
        F.when(
            (F.col("indice_educacao") < 0)
            | (F.col("indice_educacao") > 1),
            1
        ).otherwise(0)
    )
)

4. Validar e salvar

In [0]:
# Quantidade de registros marcados pelas principais regras de qualidade
df_idh_silver.select(
    F.sum("flag_chave_duplicada").alias("registros_duplicados"),
    F.sum("flag_indice_educacao_invalido").alias("indice_educacao_invalido")
).show()

+--------------------+------------------------+
|registros_duplicados|indice_educacao_invalido|
+--------------------+------------------------+
|                 116|                      19|
+--------------------+------------------------+



In [0]:
# Base WHO tratada como tabela Delta na camada Silver
df_who_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.who_life_expectancy")

In [0]:
# Base IDH tratada como tabela Delta na camada Silver
df_idh_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.idh")

In [0]:
# Confirma a criação das tabelas da camada Silver
spark.sql("SHOW TABLES IN workspace.silver").show()

+--------+-------------------+-----------+
|database|          tableName|isTemporary|
+--------+-------------------+-----------+
|  silver|                idh|      false|
|  silver|who_life_expectancy|      false|
+--------+-------------------+-----------+



A avaliação identificou diferentes níveis de completude entre os atributos e problemas de unicidade da chave composta país-ano na base de desenvolvimento humano. Também foi verificada a consistência da variável status. Os problemas identificados serão tratados ou sinalizados na camada Silver, enquanto os dados originais permanecerão preservados na Bronze para garantir a rastreabilidade do pipeline.